# 01. Data preparation

Load `merged_data.csv`, validate the time series, split by source file, and build scaled/PCA sequence datasets.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import os

config = {
    # ============================================================
    # 데이터 / 예측 길이
    # ============================================================
    "input_len": 400,
    "model_output_len": 100,   # 학습 시 한 번에 예측하는 길이
    "eval_output_len": 200,    # test AR 최종 예측 길이: 100-step × 2회
    "metric_output_len": 200,
    "stride": 100,

    # 현재 상태 변수와 최종 보고 변수
    "state_cols": ["heave", "surge"],
    "target_cols": ["heave", "surge"],
    "report_col": "heave",

    "downsample_factor": 1,

    # ============================================================
    # PCA
    # X에만 적용하며 y에는 적용하지 않음
    # 누적 설명 분산이 95% 이상이 될 때까지 주성분을 선택
    # ============================================================
    "use_pca": True,
    "pca_n_components": 0.95,

    # ============================================================
    # 실험 이름 / 모델
    # ============================================================
    "experiment_name": "lstm_transformer_db",
    "model_names": ["LSTM", "Transformer"],

    # 모델 공통 기본값
    "hidden_size": 16,
    "num_layers": 2,

    # Transformer
    "nhead": 2,
    "dim_feedforward": 64,
    "dropout": 0.0,

    # 최종 학습
    "lr": 0.001,
    "epoch": 10000,
    "batch_size": 64,
    "grad_clip": 1.0,
    "patience": 10,

    # TCN
    "tcn_channels": [32, 32, 64],
    "tcn_kernel_size": 3,
    "tcn_dropout": 0.1,

    # ============================================================
    # Loss
    # mse, ep, dilate, tildeq, db, kmbdf, cp
    # ============================================================
    "loss_type": "db",
    "peak_percentile": 90,
    "t_peak": None,

    # 전체 2변수 loss에 추가할 heave 보조 MSE 가중치
    "heave_loss_weight": 1.0,

    # MSE + EP loss
    "lambda_peak": 0.05,
    "fu": 1.4,
    "fo": 1.4,

    # TILDE-Q-like loss
    "lambda_tildeq": 0.02,
    "lambda_amp": 0.05,
    "lambda_fft_phase": 0.01,
    "lambda_corr": 0.10,

    # Checkpoint
    "save_checkpoints": True,
    "checkpoint_root": "./loss_checkpoints",

    # Optuna
    "use_optuna_best_params": True,
    "run_optuna": True,
    "optuna_n_trials": 30,
    "optuna_epoch": 30,
    "optuna_patience": 5,

    # 결과 저장
    "results_dir": "./results_ar_eval_200",
}

# 파생 설정
config["target_dim"] = len(config["target_cols"])
config["heave_idx"] = config["target_cols"].index(config["report_col"])

assert config["eval_output_len"] == 200
assert config["eval_output_len"] % config["model_output_len"] == 0
assert set(config["target_cols"]).issubset(set(config["state_cols"]))

config["experiment_dir"] = os.path.join(
    config["results_dir"], config["experiment_name"]
)
config["optuna_best_params_path"] = os.path.join(
    config["experiment_dir"], "optuna_best_params_by_model.json"
)
config["checkpoint_dir"] = os.path.join(
    config["checkpoint_root"], config["experiment_name"]
)

os.makedirs(config["experiment_dir"], exist_ok=True)
os.makedirs(config["checkpoint_dir"], exist_ok=True)

print("입력 길이:", config["input_len"])
print("학습 출력 길이:", config["model_output_len"])
print("Test AR 출력 길이:", config["eval_output_len"])
print("AR 반복 횟수:", config["eval_output_len"] // config["model_output_len"])
print("상태 변수:", config["state_cols"])
print("학습 target:", config["target_cols"])
print("최종 평가 변수:", config["report_col"])
print("PCA 사용:", config["use_pca"], "| n_components:", config["pca_n_components"])
print("실험 경로:", config["experiment_dir"])


#1. 데이터 저장

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import torch
import torch.nn as nn
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from torch.utils.data import DataLoader, TensorDataset, Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# CSV 읽기
data_path = "merged_data.csv"
df = pd.read_csv(data_path)
# 컬럼명 변경
df = df.rename(columns={
    '0': 'time',
    '1': 'surge',
    '2': 'sway',
    '3': 'heave',
    '4': 'roll',
    '5': 'pitch',
    '6': 'yaw',
    'Source_File': 'source_file'
})

# 숫자 컬럼을 진짜 숫자로 변환
numeric_cols = ['time', 'surge', 'sway', 'heave', 'roll', 'pitch', 'yaw']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 혹시 숫자로 변환 안 된 행 제거
df = df.dropna(subset=numeric_cols)

print("shape:", df.shape)
print(df.dtypes)
print(df.head())

# time, heave 추출
time = df['time'].values
heave = df['heave'].values

# 시간 간격, 샘플링 주파수
dt = time[1] - time[0]
fs = 1 / dt

print(f"샘플 수: {len(heave)}")
print(f"총 길이: {time[-1] - time[0]:.2f} s")
print(f"샘플링 간격 dt: {dt:.4f} s")
print(f"샘플링 주파수 fs: {fs:.4f} Hz")

# 그래프
plt.figure(figsize=(12, 4))
plt.plot(time, heave)
plt.xlabel("Time (s)")
plt.ylabel("Heave")
plt.title("Raw Heave Signal")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# ── 데이터 로드 후 time / heave 추출 ─────────────────────────────
time = df["time"].astype(float).values
heave = df["heave"].astype(float).values

dt = np.mean(np.diff(time))
fs = 1.0 / dt

print("── 데이터 기본 정보 ──")
print(f"샘플 수: {len(heave)}")
print(f"총 길이: {time[-1] - time[0]:.2f} s")
print(f"샘플링 간격 dt: {dt:.4f} s")
print(f"샘플링 주파수 fs: {fs:.4f} Hz")

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

# ── 시드 고정 ─────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ============================================================
# 1. 변수 설정
# ============================================================
base_input_cols = list(config["state_cols"])
target_cols = list(config["target_cols"])
report_col = config["report_col"]

INPUT_LEN = int(config["input_len"])
MODEL_OUTPUT_LEN = int(config["model_output_len"])
EVAL_OUTPUT_LEN = int(config["eval_output_len"])
CURRENT_STRIDE = int(config["stride"])
METRIC_OUTPUT_LEN = int(config.get("metric_output_len", EVAL_OUTPUT_LEN))

assert set(target_cols).issubset(set(base_input_cols)), (
    "자유 AR에서는 target_cols가 state/input 변수에 포함되어야 합니다."
)
assert report_col in target_cols
assert EVAL_OUTPUT_LEN >= MODEL_OUTPUT_LEN
assert EVAL_OUTPUT_LEN % MODEL_OUTPUT_LEN == 0
assert METRIC_OUTPUT_LEN <= EVAL_OUTPUT_LEN

config["target_dim"] = len(target_cols)
config["heave_idx"] = target_cols.index(report_col)

print("base_input_cols:", base_input_cols)
print("target_cols:", target_cols)
print("report_col:", report_col)
print(
    "MODEL_OUTPUT_LEN:", MODEL_OUTPUT_LEN,
    "| EVAL_OUTPUT_LEN:", EVAL_OUTPUT_LEN,
    "| METRIC_OUTPUT_LEN:", METRIC_OUTPUT_LEN,
)

# ============================================================
# 2. source_file 기준 Train / Val / Test split
# ============================================================
df_sorted = df.sort_values(["source_file", "time"]).reset_index(drop=True)
source_files = sorted(df_sorted["source_file"].unique())

n_files = len(source_files)
if n_files < 3:
    raise ValueError("source_file이 최소 3개 이상 필요합니다.")

train_end = max(1, int(n_files * 0.70))
val_end = max(train_end + 1, int(n_files * 0.85))
val_end = min(val_end, n_files - 1)

train_files = source_files[:train_end]
val_files = source_files[train_end:val_end]
test_files = source_files[val_end:]

train_data = df_sorted[df_sorted["source_file"].isin(train_files)].copy()
val_data = df_sorted[df_sorted["source_file"].isin(val_files)].copy()
test_data = df_sorted[df_sorted["source_file"].isin(test_files)].copy()

print("=" * 70)
print("Train / Val / Test split")
print("train files:", len(train_files), "| rows:", len(train_data))
print("val files:", len(val_files), "| rows:", len(val_data))
print("test files:", len(test_files), "| rows:", len(test_data))

# ============================================================
# 3. 다운샘플링
# ============================================================
def downsample_df(df_in, factor, source_col="source_file", time_col="time"):
    if factor is None or factor <= 1:
        return df_in.copy()

    parts = []
    for _, group in df_in.groupby(source_col):
        group = group.sort_values(time_col).reset_index(drop=True)
        parts.append(group.iloc[::factor].reset_index(drop=True))
    return pd.concat(parts, ignore_index=True)

DOWNSAMPLE_FACTOR = int(config.get("downsample_factor", 1))
train_data = downsample_df(train_data, DOWNSAMPLE_FACTOR)
val_data = downsample_df(val_data, DOWNSAMPLE_FACTOR)
test_data = downsample_df(test_data, DOWNSAMPLE_FACTOR)

print("downsample_factor:", DOWNSAMPLE_FACTOR)
print("downsampled rows:", len(train_data), len(val_data), len(test_data))

train_proc = train_data.copy()
val_proc = val_data.copy()
test_proc = test_data.copy()
input_cols = base_input_cols.copy()
pre_pca_input_cols = input_cols.copy()
print("pre_pca_input_cols:", pre_pca_input_cols)

# ============================================================
# 5. 변수별 scaler를 연속 train 데이터에 fit
# - 각 변수는 독립적으로 [-1, 1] scaling
# ============================================================
feature_scalers = {}

for col in pre_pca_input_cols:
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train_proc[[col]].astype(float).values)
    feature_scalers[col] = scaler


def scale_dataframe(df_in, cols, scalers):
    df_out = df_in.copy()
    for col in cols:
        if col not in scalers:
            raise KeyError(f"scaler가 없는 변수입니다: {col}")
        df_out[col] = scalers[col].transform(
            df_out[[col]].astype(float).values
        ).reshape(-1)
    return df_out


def inverse_scale_feature(values_scaled, col):
    values = np.asarray(values_scaled)
    shape = values.shape
    restored = feature_scalers[col].inverse_transform(values.reshape(-1, 1))
    return restored.reshape(shape)


def inverse_scale_targets(values_scaled):
    """(..., target_dim)의 scaled target을 원래 단위로 복원."""
    values = np.asarray(values_scaled)
    if values.shape[-1] != len(target_cols):
        raise ValueError(
            f"마지막 차원은 target_dim={len(target_cols)}이어야 합니다: {values.shape}"
        )

    restored = np.empty_like(values, dtype=np.float64)
    for idx, col in enumerate(target_cols):
        restored[..., idx] = inverse_scale_feature(values[..., idx], col)
    return restored


train_scaled = scale_dataframe(train_proc, pre_pca_input_cols, feature_scalers)
val_scaled = scale_dataframe(val_proc, pre_pca_input_cols, feature_scalers)
test_scaled = scale_dataframe(test_proc, pre_pca_input_cols, feature_scalers)

print("=" * 70)
print("Variable-wise scaler ranges fitted on continuous train data")
for col in pre_pca_input_cols:
    scaler = feature_scalers[col]
    print(
        f"{col:>12s}: train min={scaler.data_min_[0]: .6f}, "
        f"train max={scaler.data_max_[0]: .6f}"
    )

# ============================================================
# 6. PCA: scaled continuous train X에만 fit
#    y(target_cols)에는 PCA를 적용하지 않음
# ============================================================
pca_model = None
use_pca = bool(config.get("use_pca", False))

if use_pca:
    requested_components = config.get("pca_n_components", 0.95)
    if isinstance(requested_components, int):
        requested_components = min(requested_components, len(pre_pca_input_cols))

    pca_model = PCA(n_components=requested_components)
    pca_model.fit(train_scaled[pre_pca_input_cols].values)

    model_input_cols = [
        f"pca_{i + 1}" for i in range(pca_model.n_components_)
    ]

    def add_pca_columns(df_scaled):
        df_out = df_scaled.copy()
        z = pca_model.transform(df_out[pre_pca_input_cols].values)
        for i, col in enumerate(model_input_cols):
            df_out[col] = z[:, i]
        return df_out

    train_model_df = add_pca_columns(train_scaled)
    val_model_df = add_pca_columns(val_scaled)
    test_model_df = add_pca_columns(test_scaled)

    print("=" * 70)
    print("PCA components:", pca_model.n_components_)
    print("Explained variance ratio:", pca_model.explained_variance_ratio_)
    print("Explained variance sum:", float(pca_model.explained_variance_ratio_.sum()))
else:
    model_input_cols = pre_pca_input_cols.copy()
    train_model_df = train_scaled
    val_model_df = val_scaled
    test_model_df = test_scaled

config["input_dim"] = len(model_input_cols)
config["target_dim"] = len(target_cols)


def model_to_pre_pca_scaled(x_model):
    """모델 입력 공간(PCA 또는 scaled feature)을 pre-PCA scaled 공간으로 변환."""
    x = np.asarray(x_model)
    shape = x.shape
    if shape[-1] != len(model_input_cols):
        raise ValueError(f"model input dim 불일치: {shape}")

    if not use_pca:
        return x.copy()

    restored = pca_model.inverse_transform(x.reshape(-1, shape[-1]))
    return restored.reshape(*shape[:-1], len(pre_pca_input_cols))


def pre_pca_to_model_scaled(x_pre):
    """pre-PCA scaled feature를 모델 입력 공간으로 변환."""
    x = np.asarray(x_pre)
    shape = x.shape
    if shape[-1] != len(pre_pca_input_cols):
        raise ValueError(f"pre-PCA input dim 불일치: {shape}")

    if not use_pca:
        return x.copy()

    transformed = pca_model.transform(x.reshape(-1, shape[-1]))
    return transformed.reshape(*shape[:-1], len(model_input_cols))

# ============================================================
# 7. Sliding window 생성
#    Train/Val: 미래 100-step × 2변수
#    Test: 미래 200-step × 2변수
# ============================================================
def create_sequences_multivariate(
    data_df,
    input_cols,
    target_cols,
    input_len=400,
    output_len=100,
    stride=100,
):
    data_df = data_df.sort_values("time").reset_index(drop=True)

    x_values = data_df[input_cols].astype(float).values
    y_values = data_df[target_cols].astype(float).values

    x_list, y_list = [], []
    last_start = len(data_df) - input_len - output_len

    for start in range(0, last_start + 1, stride):
        input_end = start + input_len
        output_end = input_end + output_len
        x_list.append(x_values[start:input_end])
        y_list.append(y_values[input_end:output_end])

    if not x_list:
        return (
            np.empty((0, input_len, len(input_cols)), dtype=np.float32),
            np.empty((0, output_len, len(target_cols)), dtype=np.float32),
        )

    return np.asarray(x_list), np.asarray(y_list)


def create_sequences_multivariate_by_file(
    data_df,
    input_cols,
    target_cols,
    input_len,
    output_len,
    stride,
):
    x_all, y_all = [], []

    for source_file, file_df in data_df.groupby("source_file"):
        x, y = create_sequences_multivariate(
            file_df,
            input_cols=input_cols,
            target_cols=target_cols,
            input_len=input_len,
            output_len=output_len,
            stride=stride,
        )
        if len(x) == 0:
            print(f"sequence 생략: {source_file} (길이 부족)")
            continue
        x_all.append(x)
        y_all.append(y)

    if not x_all:
        raise ValueError(
            "생성된 sequence가 없습니다. input_len/output_len/stride 또는 파일 길이를 확인하세요."
        )

    return np.vstack(x_all), np.vstack(y_all)


X_t, y_t = create_sequences_multivariate_by_file(
    train_model_df,
    input_cols=model_input_cols,
    target_cols=target_cols,
    input_len=INPUT_LEN,
    output_len=MODEL_OUTPUT_LEN,
    stride=CURRENT_STRIDE,
)
X_v, y_v = create_sequences_multivariate_by_file(
    val_model_df,
    input_cols=model_input_cols,
    target_cols=target_cols,
    input_len=INPUT_LEN,
    output_len=MODEL_OUTPUT_LEN,
    stride=CURRENT_STRIDE,
)
X_test, y_test = create_sequences_multivariate_by_file(
    test_model_df,
    input_cols=model_input_cols,
    target_cols=target_cols,
    input_len=INPUT_LEN,
    output_len=EVAL_OUTPUT_LEN,
    stride=CURRENT_STRIDE,
)

heave_idx = target_cols.index(report_col)
config["heave_idx"] = heave_idx
config["t_peak"] = float(
    np.percentile(
        np.abs(y_t[..., heave_idx].reshape(-1)),
        config.get("peak_percentile", 90),
    )
)

print("=" * 70)
print("Final tensors")
print("model_input_cols:", model_input_cols)
print("pre_pca_input_cols:", pre_pca_input_cols)
print("target_cols:", target_cols)
print("X_t:", X_t.shape, "| y_t:", y_t.shape)
print("X_v:", X_v.shape, "| y_v:", y_v.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)
print("t_peak (scaled heave):", config["t_peak"])

assert X_t.shape[1:] == (INPUT_LEN, config["input_dim"])
assert y_t.shape[1:] == (MODEL_OUTPUT_LEN, config["target_dim"])
assert y_test.shape[1:] == (EVAL_OUTPUT_LEN, config["target_dim"])

# 전처리 객체 저장
os.makedirs(config["experiment_dir"], exist_ok=True)
preprocessing_path = os.path.join(config["experiment_dir"], "preprocessing.joblib")
joblib.dump(
    {
        "feature_scalers": feature_scalers,
        "pca_model": pca_model,
        "base_input_cols": base_input_cols,
        "pre_pca_input_cols": pre_pca_input_cols,
        "model_input_cols": model_input_cols,
        "target_cols": target_cols,
        "report_col": report_col,
    },
    preprocessing_path,
)
print("preprocessing 저장:", preprocessing_path)

# ============================================================
# 8. Dataset / DataLoader
# ============================================================
class MotionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


train_dataset = MotionDataset(X_t, y_t)
val_dataset = MotionDataset(X_v, y_v)
test_dataset = MotionDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
)

print("=" * 70)
print("Train Dataset:", train_dataset.X.shape, train_dataset.y.shape)
print("Val Dataset:", val_dataset.X.shape, val_dataset.y.shape)
print("Test Dataset:", test_dataset.X.shape, test_dataset.y.shape)
print("=" * 70)
